In [2]:
%pip install plotly

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.9 MB 1.3 MB/s eta 0:00:08
   --- ------------------------------------ 0.8/9.9 MB 1.2 MB/s eta 0:00:08
   ---- ----------------------------------- 1.0/9.9 MB 1.2 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/9.9 MB 1.2 MB/s eta 0:00:08
   ------ --------------------------------- 1.6/9.9 MB 1.2 MB/s eta 0:00:07
   ------- -------------------------------- 1.8/9.9 MB 1.2 MB/s eta 0:00:07
   -------- ------------------------------- 2.1/9.9 MB 1.2 MB/s eta 0:00:07
   --------- ------------------------------ 2.4/9.9 MB 1.2 MB/s eta 0:00:07
   ---------- ----------------------------- 2.6/9.9 MB 1.2 MB/s eta 0:00:06
   ----------- -------------------------


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution
import plotly.graph_objects as go

In [2]:
data = pd.read_csv('xy_data.csv')
x_data = data['x'].values
y_data = data['y'].values

In [3]:
def objective_function(params):
    theta_deg, M, X = params
    
    theta_rad = np.radians(theta_deg)
    
    x_shifted = x_data - X
    y_shifted = y_data - 42
    
    x_rot = x_shifted * np.cos(theta_rad) + y_shifted * np.sin(theta_rad)
    y_rot = -x_shifted * np.sin(theta_rad) + y_shifted * np.cos(theta_rad)
    
    y_pred = np.exp(M * np.abs(x_rot)) * np.sin(0.3 * x_rot)
    
    l1_loss = np.mean(np.abs(y_rot - y_pred))
    
    return l1_loss

In [4]:
bounds = [
    (0.001, 49.999),
    (-0.049, 0.049),
    (0.001, 99.999)
]

print("Running differential evolution optimizer...")
result = differential_evolution(objective_function, bounds, seed=42)

if result.success:
    theta_opt, M_opt, X_opt = result.x
    print("\n=== Optimization Results ===")
    print(f"Optimal Theta (degrees): {theta_opt:.4f}")
    print(f"Optimal M:               {M_opt:.4f}")
    print(f"Optimal X:               {X_opt:.4f}")
    print(f"Minimum L1 Distance:     {result.fun:.6f}")
else:
    print("Optimization failed to converge.")

Running differential evolution optimizer...

=== Optimization Results ===
Optimal Theta (degrees): 30.0000
Optimal M:               0.0300
Optimal X:               55.0000
Minimum L1 Distance:     0.000003


In [ ]:
t_plot = np.linspace(6, 60, 1000)

theta_rad_opt = np.radians(theta_opt)
x_pred_curve = t_plot * np.cos(theta_rad_opt) - np.exp(M_opt * np.abs(t_plot)) * np.sin(0.3 * t_plot) * np.sin(theta_rad_opt) + X_opt
y_pred_curve = 42 + t_plot * np.sin(theta_rad_opt) + np.exp(M_opt * np.abs(t_plot)) * np.sin(0.3 * t_plot) * np.cos(theta_rad_opt)

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_data, y=y_data, mode='markers', name='Dataset Points (xy_data.csv)', marker=dict(color='gray', opacity=0.5)))
fig.add_trace(go.Scatter(x=x_pred_curve, y=y_pred_curve, mode='lines', name='Predicted Parametric Curve', line=dict(color='red', width=2)))
fig.update_layout(
    title='Parametric Curve Fitting via Inverse Rotation',
    xaxis_title='X',
    yaxis_title='Y',
    template='plotly_white'
)
fig.show()